<center>
  <h1>Chirundu Town Council CDF and Financial Dataset</h1>
  <b>CSC 4792 Group Project — Group 39</b><br/>
  University of Zambia<br/>
  September 2026
</center>

---

## Overview

This notebook sets up the environment, collects document links from saved Chirundu Town Council webpages and loads the reviewed source inventory. The raw council documents are included for extraction and cleaning.

**Source:** Saved council webpages and `data/source_inventory/download_inventory.json`.

**Reference notebook:** The numbered sections, introductory pandas loading and missing-value style are adapted from *Starter Notebook: CS1 Failure Prediction Dataset*, Lighton Phiri, July 2026 (`code-datalab26-cs1-dataset-starter-notebook.ipynb`). The council data and PDF extraction methods are specific to this project.

---

## 1. Environment Setup

We import the project libraries for tables, PDF extraction and OCR. The PDF and OCR imports prepare the environment for later sections.

Run this notebook from the project folder or its `notebooks` subfolder. Install the packages listed in `src/requirements-cdf-pilot.txt` if needed. Sections 1 and 2 need the saved webpages and inventory in `data/source_inventory/`; downloaded documents and an OCR model are not needed at this stage.

Run the cells from top to bottom.


In [ ]:
# Import libraries
from pathlib import Path
from html.parser import HTMLParser
from urllib.parse import urljoin
import json
import re
import pandas as pd
import pdfplumber
import pypdfium2 as pdfium
from pypdf import PdfReader
from rapidocr_onnxruntime import RapidOCR
from IPython.display import display

pd.set_option('display.max_colwidth', 70)

# Update ROOT if the project is stored somewhere else
ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent

print('All libraries imported successfully.')
print('Project folder:', ROOT)

## 2. Collect Document Links and Load the Inventory

The council webpages were saved as HTML before extraction. We read their links, keep document downloads, combine duplicate URLs and record each source page. The parser below was previously in a separate script and is now part of this notebook.

The observed links were reviewed and organised into `download_inventory.json`, with a stable inventory number, category and filename for each selected source. This was a manual source-selection step. Documents can be downloaded using the supplied `data/source_inventory/download_documents.ps1`, which skips existing files. That collection script remains part of the codebase; it is not needed to run extraction once the files are present.

Re-running this section reads the saved pages rather than changing the collection to whatever is currently online. The URLs and page snapshots document where the files came from. The notebook does not require any local Python module.

The inventory records the selected documents and their source URLs.

In [ ]:
class Links(HTMLParser):
    def __init__(self):
        super().__init__()
        self.links = []
        self.href = None
        self.parts = []
    def handle_starttag(self, tag, attrs):
        if tag == 'a':
            self.href = dict(attrs).get('href')
            self.parts = []
    def handle_data(self, data):
        if self.href is not None:
            self.parts.append(data)
    def handle_endtag(self, tag):
        if tag == 'a' and self.href is not None:
            self.links.append((' '.join(' '.join(self.parts).split()), self.href))
            self.href = None

In [ ]:
# Read links from the saved official webpages
var_source_folder = ROOT / 'data/source_inventory'
var_base_url = 'https://www.chirunducouncil.gov.zm/'
var_observed = {}

for var_page_path in sorted(var_source_folder.glob('*.html')):
    var_page_id = '959' if var_page_path.stem == 'publications' else var_page_path.stem.removeprefix('page-')
    var_source_url = var_base_url + '?page_id=' + var_page_id
    var_parser = Links()
    var_parser.feed(var_page_path.read_text(encoding='utf-8-sig'))

    for var_label, var_href in var_parser.links:
        var_url = urljoin(var_source_url, var_href)
        if '/wp-content/uploads/' not in var_url:
            continue
        if not var_url.lower().split('?')[0].endswith(('.pdf', '.xls', '.xlsx', '.doc', '.docx', '.csv')):
            continue
        var_key = var_url.removeprefix('http://').removeprefix('https://')
        if var_key not in var_observed:
            var_observed[var_key] = {'url': var_url, 'labels': [], 'source_pages': []}
        if var_label and var_label not in var_observed[var_key]['labels']:
            var_observed[var_key]['labels'].append(var_label)
        if var_source_url not in var_observed[var_key]['source_pages']:
            var_observed[var_key]['source_pages'].append(var_source_url)

var_links_df = pd.DataFrame(var_observed.values())
print('Unique document links found:', len(var_links_df))
display(var_links_df.head())

In [ ]:
var_inventory_file = ROOT / 'data/source_inventory/download_inventory.json'
var_inventory = json.loads(var_inventory_file.read_text(encoding='utf-8'))
var_inventory_df = pd.DataFrame(var_inventory)
var_selected_ids = [1, 2, 3, 59, 5, 104, 34, 33, 32, 30, 39, 38, 35, 31, 36, 75]
display(var_inventory_df.loc[var_inventory_df['inventory_id'].isin(var_selected_ids),
                             ['inventory_id', 'display_title', 'url']])